# 01 — TextCNN

Deep-learning baseline cho hai tác vụ:

- Task 1: `NON-TOXIC` vs `TOXIC`
- Task 2: `EXPLICIT` vs `IMPLICIT` trên gold TOXIC

Thiết kế:
- vocabulary chỉ từ Train;
- class weights chỉ từ Train;
- **max epochs = 20**;
- **early stopping patience = 4**;
- chọn best checkpoint bằng **Dev Macro-F1**;
- sau training load lại checkpoint tốt nhất rồi mới Test;
- Test chỉ dùng để báo cáo cuối.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install "pandas==2.2.2" "scikit-learn==1.6.1" "tqdm==4.67.1"

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.5 MB/s eta 0:00:00


In [2]:
import os, gc, json, time, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    average_precision_score,
)
from tqdm.auto import tqdm

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Hãy bật GPU."

ROOT = Path("/content/drive/MyDrive/DS108_Toxic_4Models")
SPLIT_DIR = ROOT / "data_splits"
ARTIFACT_DIR = ROOT / "artifacts"
RESULT_DIR = ROOT / "results"
PRED_DIR = RESULT_DIR / "predictions"
CM_DIR = RESULT_DIR / "confusion_matrices"

for p in [ARTIFACT_DIR, RESULT_DIR, PRED_DIR, CM_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert (SPLIT_DIR / "train.csv").exists(), "Chạy 00_Prepare_Data.ipynb trước."

MAX_EPOCHS = 20
PATIENCE = 4
LR = 1e-3
EMBED_DIM = 200
FILTERS = 100
KERNELS = [2, 3, 4, 5]
DROPOUT = 0.5
MIN_FREQ = 2
MAX_VOCAB = 100_000

TASKS = {
    "task1": {
        "label_col": "toxicity",
        "label2id": {"NON-TOXIC": 0, "TOXIC": 1},
        "id2label": {0: "NON-TOXIC", 1: "TOXIC"},
    },
    "task2": {
        "label_col": "expression_type",
        "label2id": {"EXPLICIT": 0, "IMPLICIT": 1},
        "id2label": {0: "EXPLICIT", 1: "IMPLICIT"},
    },
}

def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

def get_task(split_name, task):
    x = pd.read_csv(SPLIT_DIR / f"{split_name}.csv")
    if task == "task2":
        x = x[x["toxicity"].eq("TOXIC")].copy()
    x["label_id"] = x[TASKS[task]["label_col"]].map(TASKS[task]["label2id"])
    assert x["label_id"].notna().all()
    return x.reset_index(drop=True)

def class_weights(train_frame):
    c = train_frame["label_id"].value_counts().sort_index()
    n, k = len(train_frame), len(c)
    return torch.tensor([n / (k * c.loc[i]) for i in range(k)], dtype=torch.float32)

def metrics(y, yp, probs):
    acc = accuracy_score(y, yp)
    mp, mr, mf, _ = precision_recall_fscore_support(
        y, yp, average="macro", zero_division=0
    )
    _, _, wf, _ = precision_recall_fscore_support(
        y, yp, average="weighted", zero_division=0
    )
    pr_auc = average_precision_score(y, probs[:, 1])
    return {
        "accuracy": float(acc),
        "macro_precision": float(mp),
        "macro_recall": float(mr),
        "macro_f1": float(mf),
        "weighted_f1": float(wf),
        "pr_auc_positive_class": float(pr_auc),
    }

def save_json(obj, path):
    Path(path).write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

In [3]:
PAD, UNK = "<PAD>", "<UNK>"

def tokens(text):
    return str(text).strip().split()

def build_vocab(texts):
    counter = Counter()
    for text in texts:
        counter.update(tokens(text))
    vocab = {PAD: 0, UNK: 1}
    for tok, freq in counter.most_common():
        if freq < MIN_FREQ:
            break
        if len(vocab) >= MAX_VOCAB:
            break
        vocab[tok] = len(vocab)
    return vocab

class TextDataset(Dataset):
    def __init__(self, frame, vocab, max_len):
        self.frame = frame.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        ids = [self.vocab.get(t, 1) for t in tokens(row["comment"])][:self.max_len]
        ids += [0] * (self.max_len - len(ids))
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels": torch.tensor(int(row["label_id"]), dtype=torch.long),
        }

class TextCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(EMBED_DIM, FILTERS, k) for k in KERNELS
        ])
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(FILTERS * len(KERNELS), 2)

    def forward(self, input_ids):
        x = self.embedding(input_ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            z = F.relu(conv(x))
            pooled.append(F.max_pool1d(z, z.size(2)).squeeze(2))
        x = torch.cat(pooled, dim=1)
        return self.fc(self.dropout(x))

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    losses, logits_all, y_all = [], [], []
    for batch in loader:
        x = batch["input_ids"].to(DEVICE)
        y = batch["labels"].to(DEVICE)
        logits = model(x)
        losses.append(loss_fn(logits, y).item())
        logits_all.append(logits.cpu())
        y_all.append(y.cpu())

    logits = torch.cat(logits_all)
    y = torch.cat(y_all).numpy()
    probs = torch.softmax(logits, dim=1).numpy()
    yp = probs.argmax(axis=1)

    m = metrics(y, yp, probs)
    m["loss"] = float(np.mean(losses))
    return m, y, yp, probs

In [4]:
def train_task(task):
    out_dir = ARTIFACT_DIR / "textcnn" / task
    out_dir.mkdir(parents=True, exist_ok=True)
    pred_path = PRED_DIR / f"textcnn_{task}.csv"

    if (out_dir / "test_metrics.json").exists() and pred_path.exists():
        print("SKIP completed:", task)
        return

    train_df = get_task("train", task)
    dev_df = get_task("dev", task)
    test_df = get_task("test", task)

    vocab = build_vocab(train_df["comment"])
    train_lens = train_df["comment"].map(lambda x: len(tokens(x)))
    max_len = min(200, max(max(KERNELS), int(np.quantile(train_lens, 0.99))))
    batch_size = 64 if task == "task1" else 32

    loaders = {}
    for split_name, frame, shuffle in [
        ("train", train_df, True),
        ("dev", dev_df, False),
        ("test", test_df, False),
    ]:
        loaders[split_name] = DataLoader(
            TextDataset(frame, vocab, max_len),
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=2,
            pin_memory=True,
        )

    seed_all()
    model = TextCNN(len(vocab)).to(DEVICE)
    w = class_weights(train_df).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_dev_f1 = -1.0
    best_epoch = None
    wait = 0
    history = []
    best_path = out_dir / "best_model.pt"
    started = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []

        for batch in tqdm(loaders["train"], desc=f"TextCNN {task} epoch {epoch}", leave=False):
            x = batch["input_ids"].to(DEVICE)
            y = batch["labels"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        dev_m, _, _, _ = evaluate(model, loaders["dev"], loss_fn)
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "dev_loss": dev_m["loss"],
            "dev_accuracy": dev_m["accuracy"],
            "dev_macro_precision": dev_m["macro_precision"],
            "dev_macro_recall": dev_m["macro_recall"],
            "dev_macro_f1": dev_m["macro_f1"],
            "dev_pr_auc": dev_m["pr_auc_positive_class"],
        }
        history.append(row)
        print(row)

        if dev_m["macro_f1"] > best_dev_f1 + 1e-6:
            best_dev_f1 = dev_m["macro_f1"]
            best_epoch = epoch
            wait = 0
            torch.save(model.state_dict(), best_path)
        else:
            wait += 1
            if wait >= PATIENCE:
                print("Early stopping.")
                break

    training_seconds = time.time() - started
    pd.DataFrame(history).to_csv(out_dir / "training_history.csv", index=False)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    dev_m, _, _, _ = evaluate(model, loaders["dev"], loss_fn)
    test_m, y, yp, probs = evaluate(model, loaders["test"], loss_fn)

    id2label = TASKS[task]["id2label"]
    pred = test_df[["row_id", "comment"]].copy()
    pred["gold_label"] = [id2label[int(i)] for i in y]
    pred["predicted_label"] = [id2label[int(i)] for i in yp]
    pred["prob_class_0"] = probs[:, 0]
    pred["prob_class_1"] = probs[:, 1]
    pred["correct"] = pred["gold_label"].eq(pred["predicted_label"])
    pred.to_csv(pred_path, index=False)

    p, r, f, s = precision_recall_fscore_support(
        y, yp, labels=[0, 1], average=None, zero_division=0
    )
    pd.DataFrame([
        {"label": id2label[i], "precision": p[i], "recall": r[i], "f1": f[i], "support": s[i]}
        for i in [0, 1]
    ]).to_csv(out_dir / "classification_report.csv", index=False)

    cm = confusion_matrix(y, yp, labels=[0, 1])
    pd.DataFrame(
        cm,
        index=[id2label[0], id2label[1]],
        columns=[id2label[0], id2label[1]],
    ).to_csv(CM_DIR / f"textcnn_{task}_cm.csv")

    save_json(dev_m, out_dir / "dev_metrics.json")
    save_json(test_m, out_dir / "test_metrics.json")
    save_json({
        "model": "TextCNN",
        "task": task,
        "seed": SEED,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": PATIENCE,
        "best_epoch": best_epoch,
        "batch_size": batch_size,
        "max_len": max_len,
        "vocab_size": len(vocab),
        "learning_rate": LR,
        "class_weights": w.detach().cpu().tolist(),
        "training_time_seconds": training_seconds,
        "gpu": torch.cuda.get_device_name(0),
    }, out_dir / "experiment_config.json")

    (out_dir / "vocab.json").write_text(
        json.dumps(vocab, ensure_ascii=False),
        encoding="utf-8"
    )

    print("BEST EPOCH:", best_epoch)
    print("TEST:", test_m)

    del model, loaders
    gc.collect()
    torch.cuda.empty_cache()

for task in ["task1", "task2"]:
    train_task(task)

TextCNN task1 epoch 1:   0%|          | 0/609 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.6328730091868559, 'dev_loss': 0.5108858650767958, 'dev_accuracy': 0.8619272652558044, 'dev_macro_precision': 0.6877058047316669, 'dev_macro_recall': 0.7234093737468439, 'dev_macro_f1': 0.7028587709167502, 'dev_pr_auc': 0.48630670320909575}


TextCNN task1 epoch 2:   0%|          | 0/609 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.4861317334699709, 'dev_loss': 0.4743046172253497, 'dev_accuracy': 0.7530306143414834, 'dev_macro_precision': 0.6324390616083266, 'dev_macro_recall': 0.7712385785911715, 'dev_macro_f1': 0.639290312237952, 'dev_pr_auc': 0.5597908580648574}


TextCNN task1 epoch 3:   0%|          | 0/609 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.40595875506722084, 'dev_loss': 0.5252515348521146, 'dev_accuracy': 0.7041298541195808, 'dev_macro_precision': 0.6194487988199613, 'dev_macro_recall': 0.7655347503414165, 'dev_macro_f1': 0.6053442459888423, 'dev_pr_auc': 0.583228820752679}


... (24 outputs left out)

TextCNN task2 epoch 4:   0%|          | 0/147 [00:00<?, ?it/s]

{'epoch': 4, 'train_loss': 0.3152912047241821, 'dev_loss': 1.3376300899606002, 'dev_accuracy': 0.8464163822525598, 'dev_macro_precision': 0.6479774705581157, 'dev_macro_recall': 0.5537674418604651, 'dev_macro_f1': 0.5627300766092926, 'dev_pr_auc': 0.3184809658591554}


TextCNN task2 epoch 5:   0%|          | 0/147 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.22801863779725673, 'dev_loss': 1.5188248722176803, 'dev_accuracy': 0.8515358361774744, 'dev_macro_precision': 0.675445570794408, 'dev_macro_recall': 0.5615813953488372, 'dev_macro_f1': 0.5739677605354859, 'dev_pr_auc': 0.3316492826257152}
Early stopping.
BEST EPOCH: 1
TEST: {'accuracy': 0.7150170648464164, 'macro_precision': 0.6003578464878774, 'macro_recall': 0.6741395348837209, 'macro_f1': 0.6012582152737881, 'weighted_f1': 0.7517251920248044, 'pr_auc_positive_class': 0.35018854262587956, 'loss': 0.6063241174346522}
